# Despliegue en Colab (GPU T4 + túnel público)

Pone en línea `final/app.py` con una URL pública. Hugging Face ya no da GPU gratuita suficiente (ZeroGPU limita el uso) y un
Space de CPU tardaría unos dos minutos por consulta, así que la app corre en la GPU de Colab y se abre al exterior con un
túnel de cloudflared (no pide cuenta).

**Antes de ejecutar**
1. Entorno de ejecución: **GPU T4** (Entorno de ejecución, Cambiar tipo de entorno).
2. Ejecutar todas las celdas en orden. Aparecen dos URL en la celda 3: la de la demo y la del panel de MLflow
   (observabilidad: latencias y válvula por petición). No hace falta ninguna cuenta ni clave.

**Tener en cuenta**
- La URL cambia cada vez que se reinicia la sesión y deja de funcionar si la sesión de Colab se cierra. La celda 5 debe
  quedar ejecutándose mientras se use la demo, con la pestaña abierta.
- Colab gratuito puede desconectar por inactividad o por límite de uso de GPU. Conviene abrir la URL unos minutos antes de la
  sustentación y tener el video de respaldo.

## 1 · Preparar el entorno

In [ ]:
import os, subprocess, sys
from importlib.metadata import PackageNotFoundError, version as version_instalada
from pathlib import Path
from packaging.version import Version
import torch

assert torch.cuda.is_available(), 'No hay GPU: cambie el entorno de ejecución a GPU T4.'
print('GPU:', torch.cuda.get_device_name(0))

REPO_URL = 'https://github.com/Bosnape/cabrejos-ortiz-valencia-lopez.git'
RAIZ = Path('/content/cabrejos-ortiz-valencia-lopez')
if not RAIZ.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(RAIZ)], check=True)
else:
    subprocess.run(['git', '-C', str(RAIZ), 'pull', '--ff-only'], check=True)
os.chdir(RAIZ)

# PEFT rechaza el torchao viejo de algunos runtimes de Colab.
try:
    torchao = version_instalada('torchao')
except PackageNotFoundError:
    torchao = None
if torchao is not None and Version(torchao) < Version('0.16.0'):
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)

pillow_fija = version_instalada('pillow')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'final/requirements.txt', f'pillow=={pillow_fija}'], check=True)
print('Listo.')

## 2 · Arrancar la app

Se carga una sola vez, en segundo plano. Mientras carga, `/health` responde `cargando`; cuando termina responde `ok`.

In [ ]:
import time, requests

MLFLOW_DB = 'sqlite:///' + str(RAIZ / 'mlflow.db')
entorno = dict(os.environ, PORT='7860', LOG_PETICIONES=str(RAIZ / 'peticiones.jsonl'), MLFLOW_TRACKING_URI=MLFLOW_DB)

# Panel de MLflow en el puerto 5000, sobre la misma base que escribe la app. Se arranca PRIMERO y se espera a que
# cree el esquema; si la app y el panel lo crean a la vez, SQLite falla con "table ... already exists".
for previo in ('ui', 'app'):                            # por si se vuelve a ejecutar la celda
    if previo in globals():
        globals()[previo].terminate()
time.sleep(2)
for archivo in ('mlflow.db', 'mlflow.db-wal', 'mlflow.db-shm'):
    (RAIZ / archivo).unlink(missing_ok=True)

entorno_ui = dict(entorno, MLFLOW_SERVER_DISABLE_SECURITY_MIDDLEWARE='true')
ui = subprocess.Popen([sys.executable, '-m', 'mlflow', 'server', '--host', '127.0.0.1', '--port', '5000',
                       '--backend-store-uri', MLFLOW_DB], env=entorno_ui,
                      stdout=open('mlflow.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(90):
    if ui.poll() is not None:
        raise RuntimeError('El panel de MLflow se cerró. Revise mlflow.log:\n' + open('mlflow.log').read()[-2000:])
    try:
        if requests.get('http://127.0.0.1:5000/health', timeout=2).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError('El panel de MLflow no arrancó. Revise mlflow.log.')

app_log = open('app.log', 'w')
app = subprocess.Popen([sys.executable, 'final/app.py'], env=entorno, stdout=app_log, stderr=subprocess.STDOUT)

salud = {}
for _ in range(180):                                   # hasta 6 minutos
    if app.poll() is not None:
        raise RuntimeError('La app se cerró. Revise app.log:\n' + open('app.log').read()[-2000:])
    try:
        salud = requests.get('http://127.0.0.1:7860/health', timeout=2).json()
        if salud.get('status') in ('ok', 'error', 'degradado'):
            break
    except Exception:
        pass
    time.sleep(2)
print(salud)
assert salud.get('status') == 'ok', 'La app no quedó lista. Revise app.log.'

## 3 · Abrir la app y el panel al exterior

Descarga cloudflared y abre dos túneles rápidos: uno hacia la app (puerto 7860) y otro hacia el panel de MLflow (puerto 5000).
Las direcciones terminan en `trycloudflare.com`.

In [ ]:
import re
subprocess.run(['curl', '-sL', '-o', 'cloudflared', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
os.chmod('cloudflared', 0o755)

def abrir_tunel(puerto, nombre, extra=()):
    log = f'cloudflared_{nombre}.log'
    proc = subprocess.Popen(['./cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{puerto}', '--no-autoupdate', *extra],
                            stdout=open(log, 'w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open(log).read())
        if m:
            return proc, m.group(0)
        time.sleep(1)
    raise RuntimeError(f'No apareció la URL del túnel. Revise {log}:\n' + open(log).read()[-1500:])

tunel, URL = abrir_tunel(7860, 'app')
tunel_ui, URL_MLFLOW = abrir_tunel(5000, 'mlflow', ('--http-host-header', '127.0.0.1:5000'))
print('URL de la demo:    ', URL)
print('Health check:      ', URL + '/health')
print('Panel de MLflow:   ', URL_MLFLOW)

## 4 · Comprobar desde afuera

Pasa por el túnel, es decir, por la ruta pública, igual que lo haría otra persona. La dirección puede tardar unos segundos
en responder después de crearse. Después, abrir la URL desde el celular con datos móviles.

In [ ]:
for _ in range(30):
    try:
        r = requests.get(URL + '/health', timeout=10)
        if r.status_code == 200:
            print(r.json()); break
    except Exception:
        pass
    time.sleep(3)
else:
    raise RuntimeError('La URL pública no responde todavía; revise cloudflared.log.')

from gradio_client import Client
cliente = Client(URL, verbose=False)
t = time.time()
resp = cliente.predict('Me despidieron estando embarazada y no hubo autorización del Ministerio. ¿Qué normas debo revisar?',
                       api_name='/responder')
print(f'{time.time() - t:.1f} s\n', resp[:600])
print('\nPanel MLflow, estado HTTP:', requests.get(URL_MLFLOW, timeout=20).status_code)
print('\nÚltima línea del log:\n', open(RAIZ / 'peticiones.jsonl').read().splitlines()[-1])

## 5 · Mantenerla en línea

Dejar esta celda ejecutándose mientras se use la demo. Imprime cada minuto el estado de `/health`. Para apagar todo,
interrumpir la celda y ejecutar la última.

In [ ]:
try:
    while True:
        try:
            s = requests.get(URL + '/health', timeout=10).json()
            print(time.strftime('%H:%M:%S'), s['status'], '| peticiones:', s['peticiones'], '| válvula:', s['valvula'],
                  '| errores:', s['errores'], '| MLflow:', s['mlflow'])
        except Exception as e:
            print(time.strftime('%H:%M:%S'), 'sin respuesta:', type(e).__name__)
        time.sleep(60)
except KeyboardInterrupt:
    print('Interrumpido; la app y el túnel siguen corriendo.')

## 6 · Apagar

In [ ]:
tunel.terminate(); tunel_ui.terminate(); ui.terminate(); app.terminate()
print('App, panel y túneles detenidos.')